In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical

import warnings
warnings.filterwarnings("ignore")

In [ ]:
#export the dataset
train = pd.read_csv('/kaggle/input/forest-cover-type-prediction/train.csv', dtype=str)
test = pd.read_csv( '/kaggle/input/forest-cover-type-prediction/test.csv', dtype=str)

In [ ]:
X_train = train.drop(columns = ['Id', 'Cover_Type']) 
X_test = test.drop(columns = ['Id'], axis=1)
y_train = train['Cover_Type'].astype(int)
print("Training features shape:", X_train.shape)
print("Training label shape:", y_train.shape)

In [ ]:
df = train.drop(['Id'], axis = 1)
df

In [ ]:
train.head()

In [ ]:
# numeric features from training set
num_features = train.iloc[:,1:11]
num_features.describe()

In [ ]:
# categorical features from training set
cat_features = train.iloc[:, 11:-1]
cat_features.describe()

In [ ]:

correlationMatrix = df.corr()
correlationAbs = correlationMatrix.abs().unstack()
correlationSorted = correlationAbs.sort_values(ascending = False).drop_duplicates()

correlationSorted.head(54)

In [ ]:
# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)

In [ ]:
# One-hot encode the labels
y_train_encoded = to_categorical(y_train - 1)  # subtract 1 to make labels start from 0

In [ ]:

model = Sequential([
    Dense(64, activation='relu'),
    Dense(7, activation='softmax')
])

In [ ]:
# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
#train the model
history = model.fit(X_train_scaled, y_train_encoded, batch_size=32, epochs=100)

In [ ]:
# Aply the scaling to the test set features
scaler = StandardScaler()
scaler.fit(X_test)
# Make predictions
X_test_scaled = scaler.transform(X_test.values)
predictions = model.predict(X_test_scaled)
# Add 1 to convert back to class labels starting from 1
predicted_labels = np.argmax(predictions, axis = 1) + 1 

In [ ]:
# Create submission DataFrame
submission = pd.DataFrame({'Id': test['Id'], 'Cover_Type': predicted_labels})
submission.head()

In [ ]:
# Save the submission file
submission.to_csv('submission.csv', index=False)